# 01 — Línea base con InstructPix2Pix

Este notebook carga el modelo base `timbrooks/instruct-pix2pix` y realiza una edición de ejemplo. Sirve para:

1. Verificar que el entorno y dependencias funcionan.
2. Establecer una línea base cualitativa antes de entrenar.
3. Explorar el efecto de `guidance_scale` e `image_guidance_scale`.

In [ ]:
import torch
from diffusers import StableDiffusionInstructPix2PixPipeline
from diffusers.utils import load_image
from PIL import Image
import matplotlib.pyplot as plt

## 1. Cargar el modelo

In [ ]:
model_id = "timbrooks/instruct-pix2pix"

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.float16
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float32  # MPS suele tener problemas con float16
else:
    device = "cpu"
    dtype = torch.float32

print(f"Dispositivo: {device}")

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    safety_checker=None,
)

# Si la memoria es justa, descomenta una de estas líneas:
# pipe.enable_model_cpu_offload()
# pipe.enable_sequential_cpu_offload()

pipe = pipe.to(device)

## 2. Cargar imagen de ejemplo

In [ ]:
url = "https://raw.githubusercontent.com/timothybrooks/instruct-pix2pix/main/imgs/example.jpg"
image = load_image(url).convert("RGB")
image

## 3. Editar imagen

In [ ]:
prompt = "turn him into a cyborg"
num_inference_steps = 20
guidance_scale = 7.5
image_guidance_scale = 1.5
seed = 42

generator = torch.Generator(device).manual_seed(seed) if seed else None

edited = pipe(
    prompt,
    image=image,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    image_guidance_scale=image_guidance_scale,
    generator=generator,
).images[0]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(edited)
axes[1].set_title(f"Editado: {prompt}")
axes[1].axis("off")
plt.show()

## 4. Explorar guidance scales

In [ ]:
image_guidance_values = [1.0, 1.5, 2.0]
results = []

for igs in image_guidance_values:
    generator = torch.Generator(device).manual_seed(seed)
    out = pipe(
        prompt,
        image=image,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        image_guidance_scale=igs,
        generator=generator,
    ).images[0]
    results.append((igs, out))

fig, axes = plt.subplots(1, len(results), figsize=(16, 5))
for ax, (igs, out) in zip(axes, results):
    ax.imshow(out)
    ax.set_title(f"image_guidance_scale={igs}")
    ax.axis("off")
plt.show()

## 5. Guardar resultado

In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)
edited.save("../outputs/baseline_edit.jpg")
print("Guardado en ../outputs/baseline_edit.jpg")